============================================
Brasil Olímpico - Scripts do SSMS -> JSON do site
Roda no Google Colab
============================================
Passo 1: No SSMS, clique com botão direito no banco jogosolimpicos
  Tarefas > Gerar Scripts > Tabelas + Dados > Salvar como .sql
Passo 2: No Colab, suba esse .sql (botão de upload no painel esquerdo)
Passo 3: Rode as células abaixo, na ordem
Passo 4: Baixe o dados_brasil_olimpico.json, o index.html E o index_standalone.html atualizados e suba os 3 no GitHub

Por que 3 arquivos? O site tem 2 versões do HTML, do mesmo JSON:
  1) dados_brasil_olimpico.json (usado pela versão leve no GitHub Pages)
  2) index.html — versão LEVE (~324 KB, sem dados embutidos, busca o JSON com fetch). Para GitHub Pages / servidor.
  3) index_standalone.html — versão STANDALONE (1 arquivo só, com os dados embutidos em <script id="embedded-data">). Para dois cliques / WhatsApp / e-mail.
As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual. A única diferença é onde o dado mora.

In [ ]:
# --- CÉLULA 1: Upload do .sql ---
from google.colab import files
import re, json, collections

print("Selecione o arquivo .sql gerado pelo SSMS")
uploaded = files.upload()  # vai abrir a janela de upload
nome_arquivo = list(uploaded.keys())[0]
print(f"Arquivo recebido: {nome_arquivo}")


In [ ]:
# --- CÉLULA 2: Parse do .sql ---
# O SSMS gera em UTF-16. Tente UTF-16, se falhar, tente UTF-8.
raw = open(nome_arquivo, 'rb').read()
try:
    t = raw.decode('utf-16')
except:
    t = raw.decode('utf-8', errors='ignore')

lines = [l for l in t.splitlines() if l.startswith("INSERT [jogosolimpicos].")]

def parse_values(line):
    m = re.search(r'VALUES \((.*)\)\s*$', line)
    s = m.group(1)
    parts = []; cur = ""; in_str = False; i = 0
    while i < len(s):
        ch = s[i]
        if ch == "'":
            if i+1 < len(s) and s[i+1] == "'":
                cur += "''"; i += 2; continue
            in_str = not in_str; cur += ch; i += 1; continue
        if ch == "," and not in_str:
            parts.append(cur.strip()); cur = ""; i += 1; continue
        cur += ch; i += 1
    parts.append(cur.strip())
    def clean(v):
        v = v.strip()
        if v == "NULL": return None
        mm = re.match(r"CAST\(N'(.*)' AS Date\)", v)
        if mm: return mm.group(1).replace("''", "'")
        if v.startswith("N'") and v.endswith("'"): return v[2:-1].replace("''", "'")
        try: return int(v)
        except: return v
    return [clean(x) for x in parts]

atletas = {}   # id -> dict
edicoes = {}   # id -> dict
medalhas_map = {}  # id -> dict
rel = []       # lista de [ID, MedalhaID, AtletaID, EdicaoID]

for l in lines:
    if ".[atleta]" in l:
        r = parse_values(l)
        # [AtletaID, Nome_Principal, Nome_Complemento, Sexo, Data_Nascimento, Data_Morte, Cidade, Estado, Pais]
        atletas[r[0]] = {"id": r[0], "nome": r[1], "nome_oficial": r[2], "sexo": r[3],
                         "nasc": r[4], "cidade": r[6], "estado": r[7], "pais": r[8]}
    elif ".[edicao]" in l:
        r = parse_values(l)
        # [EdicaoID, Cidade, Pais, Ano, Verao_Inverno, Abertura, Encerramento, Inicio, Fim]
        edicoes[r[0]] = {"id": r[0], "cidade": r[1], "pais": r[2], "ano": r[3], "estacao": r[4]}
    elif ".[medalha]" in l:
        r = parse_values(l)
        # [MedalhaID, EdicaoID, Data, Medalha, Individual_Coletivo, Modalidade, Evento, Genero]
        medalhas_map[r[0]] = {"id": r[0], "edicao_id": r[1], "data": r[2], "tipo": r[3],
                              "indiv": r[4], "modalidade": r[5], "evento": r[6], "genero": r[7]}
    elif ".[medalhascomatletas]" in l:
        rel.append(parse_values(l))

print(f"atletas: {len(atletas)} | edicoes: {len(edicoes)} | medalhas: {len(medalhas_map)} | vínculos: {len(rel)}")


In [ ]:
# --- CÉLULA 3: Validações rápidas ---
# Se alguma falhar, pare aqui e confira o banco antes de gerar o JSON
assert len(atletas) == len(set(atletas.keys())), "ID de atleta duplicado"
assert len(medalhas_map) == len(set(medalhas_map.keys())), "ID de medalha duplicado"
atleta_ids = set(atletas.keys()); edicao_ids = set(edicoes.keys()); medalha_ids = set(medalhas_map.keys())
assert all(m["edicao_id"] in edicao_ids for m in medalhas_map.values()), "Medalha sem edição"
assert all(r[1] in medalha_ids and r[2] in atleta_ids for r in rel), "Vínculo com FK quebrada"
assert len(set((r[1], r[2]) for r in rel)) == len(rel), "Vínculo duplicado (medalha+atleta)"
print("✅ Validações OK - sem FK quebrada, sem duplicado")

# Totais por tipo
from collections import Counter
print("Totais:", Counter(m["tipo"] for m in medalhas_map.values()))


In [ ]:
# --- CÉLULA 4: Montar o JSON do site ---
# Mapa medalha -> atletas
medalha_para_atletas = collections.defaultdict(list)
for r in rel:
    medalha_para_atletas[r[1]].append(atletas[r[2]])

def genero_label(g):
    return {"M": "Masculino", "F": "Feminino", "F&M": "Misto"}.get(g, g)

medalhas = []
for mid, m in sorted(medalhas_map.items()):
    ed = edicoes[m["edicao_id"]]
    ats = medalha_para_atletas[mid]
    det = [{"nome": a["nome"], "nome_oficial": a["nome_oficial"],
            "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()} for a in ats]
    medalhas.append({
        "id": mid, "ano": ed["ano"], "cidade": ed["cidade"], "estacao": ed["estacao"],
        "modalidade": m["modalidade"], "evento": m["evento"], "genero": m["genero"],
        "genero_label": genero_label(m["genero"]), "indiv": m["indiv"], "data": m["data"],
        "medalha_tipo": m["tipo"], "atletas": [a["nome"] for a in ats],
        "atletas_detalhe": det,
        "busca_atletas": " ".join(d["busca"] for d in det)
    })

# by_edicao: TODAS as 55 edições, incluindo zeros
by_edicao = []
for eid, e in sorted(edicoes.items(), key=lambda x: (x[1]["ano"], x[1]["estacao"])):
    c = Counter(m["tipo"] for m in medalhas_map.values() if m["edicao_id"] == eid)
    by_edicao.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                      "ouro": c.get("Ouro", 0), "prata": c.get("Prata", 0), "bronze": c.get("Bronze", 0),
                      "total": sum(c.values())})

# by_modalidade
by_mod = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for m in medalhas_map.values():
    k = m["modalidade"]; by_mod[k]["total"] += 1
    by_mod[k]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_modalidade = [{"modalidade_nome": k, "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
                 for k, v in sorted(by_mod.items(), key=lambda x: -x[1]["total"])]

# by_estado: conta VÍNCULOS medalha + atleta (1 medalha coletiva conta 1x por atleta que ganhou medalha)
by_est = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    a = atletas[r[2]]; m = medalhas_map[r[1]]
    if a["pais"] != "Brasil": continue
    est = a["estado"] or "Não informado"
    by_est[est]["total"] += 1
    by_est[est]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_estado = [{"estado_nome": k, "total_vinculos": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
             for k, v in sorted(by_est.items(), key=lambda x: -x[1]["total"])]

# by_atleta + ranking
by_atl = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    m = medalhas_map[r[1]]
    by_atl[r[2]]["total"] += 1
    by_atl[r[2]]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_atleta = []
ranking = []
for aid, v in sorted(by_atl.items(), key=lambda x: (-x[1]["total"], -x[1]["ouro"], -x[1]["prata"])):
    a = atletas[aid]
    by_atleta.append({"atleta_nome": a["nome"], "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]})
    ranking.append({"nome": a["nome"], "sexo": a["sexo"], "estado": a["estado"], "pais": a["pais"], "cidade": a["cidade"],
                    "total": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"],
                    "nome_oficial": a["nome_oficial"], "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()})

# by_genero por edição (com zeros)
by_ge = collections.defaultdict(lambda: {"Masculino":0,"Feminino":0,"Misto":0})
for m in medalhas:
    by_ge[(m["ano"], m["estacao"])][m["genero_label"]] += 1
by_genero_edicao_completa = []
for e in by_edicao:
    v = by_ge.get((e["ano"], e["estacao"]), {"Masculino":0,"Feminino":0,"Misto":0})
    by_genero_edicao_completa.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                                      "Masculino": v["Masculino"], "Feminino": v["Feminino"], "Misto": v["Misto"],
                                      "total": sum(v.values())})
# Compatibilidade: by_genero_ano (agrupado por ano)
by_genero_ano = []
for ano in sorted(set(m["ano"] for m in medalhas)):
    agg = {"Masculino":0,"Feminino":0,"Misto":0}
    for m in medalhas:
        if m["ano"] == ano: agg[m["genero_label"]] += 1
    by_genero_ano.append({"ano": ano, **agg})

# --- idades_detalhe ---
# 1 linha por VÍNCULO medalha + atleta (um time de vôlei com 12 atletas vira 12 linhas)
# idade = data da medalha - data de nascimento (leva em conta ano bissexto)
from datetime import datetime

def calc_idade(nasc_str, medalha_data_str):
    if not nasc_str or not medalha_data_str:
        return None
    try:
        nasc = datetime.strptime(str(nasc_str)[:10], "%Y-%m-%d")
        med = datetime.strptime(str(medalha_data_str)[:10], "%Y-%m-%d")
    except:
        return None
    dias = (med - nasc).days
    if dias < 0:
        return None
    return dias / 365.25

idades_detalhe = []
for r in rel:
    mid, aid = r[1], r[2]
    a = atletas[aid]
    m = medalhas_map[mid]
    ed = edicoes[m["edicao_id"]]
    idade = calc_idade(a["nasc"], m["data"])
    if idade is None:
        continue  # sem data de nascimento -> pula, não quebra o JSON
    nasc_dt = datetime.strptime(str(a["nasc"])[:10], "%Y-%m-%d")
    med_dt = datetime.strptime(str(m["data"])[:10], "%Y-%m-%d")
    anos = med_dt.year - nasc_dt.year - ((med_dt.month, med_dt.day) < (nasc_dt.month, nasc_dt.day))
    try:
        aniv = nasc_dt.replace(year=nasc_dt.year + anos)
    except ValueError:
        aniv = nasc_dt.replace(year=nasc_dt.year + anos, day=28)  # quem nasceu em 29/02
    dias = (med_dt - aniv).days
    idade_texto = f"{anos} anos e {dias} dias"
    idades_detalhe.append({
        "nome": a["nome"],
        "idade": round(idade, 2),
        "idade_texto": idade_texto,
        "ano": ed["ano"],
        "medalha_tipo": m["tipo"],
        "modalidade": m["modalidade"],
        "data_medalha": m["data"],
        "data_nascimento": a["nasc"],
    })

idades = [x["idade"] for x in idades_detalhe]

print(f"idades_detalhe: {len(idades_detalhe)} linhas")
print(f"Sem nascimento (puladas): {len(rel) - len(idades_detalhe)}")
mais_jovem = min(idades_detalhe, key=lambda x: x["idade"])
mais_velho = max(idades_detalhe, key=lambda x: x["idade"])
print(f"Mais jovem: {mais_jovem['nome']} - {mais_jovem['idade_texto']} em {mais_jovem['data_medalha'][:4]}")
print(f"Mais velho: {mais_velho['nome']} - {mais_velho['idade_texto']} em {mais_velho['data_medalha'][:4]}")

dados = {"medalhas": medalhas, "by_edicao": by_edicao, "by_modalidade": by_modalidade,
         "by_estado": by_estado, "by_atleta": by_atleta, "ranking": ranking,
         "by_genero_edicao_completa": by_genero_edicao_completa, "by_genero_ano": by_genero_ano,
         "edicoes": by_edicao, "by_edicao_completa": by_edicao,
         "idades": idades, "idades_detalhe": idades_detalhe}

with open("dados_brasil_olimpico.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=2)

print(f"✅ Gerado dados_brasil_olimpico.json: {len(medalhas)} medalhas, {len(by_edicao)} edições, {len(ranking)} medalhistas")
# Validação cruzada: 3 agrupamentos diferentes têm que dar o mesmo total
soma_edicao = sum(x["total"] for x in by_edicao)
soma_genero = sum(x["total"] for x in by_genero_edicao_completa)
soma_modalidade = sum(x["total_medalhas"] for x in by_modalidade)
soma_vinculos = sum(x["total"] for x in ranking)
print(f"   Soma por edição: {soma_edicao} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma por gênero: {soma_genero} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma por modalidade: {soma_modalidade} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma vínculos medalha + atleta: {soma_vinculos} (deve ser = {len(rel)} vínculos)")
print(f"   Top atleta: {ranking[0]['nome']} com {ranking[0]['total']} medalhas")


In [ ]:
# --- CÉLULA 5: Gerar index.html (leve) + index_standalone.html (1 arquivo só) do mesmo JSON ---
# NUNCA mais gere a versão híbrida (com os 2 ao mesmo tempo).
# Como funciona:
#   - index.html (LEVE): sem dados embutidos, com o comentário
#     <!-- Dados carregados de dados_brasil_olimpico.json (versão web, sem cópia embutida para ficar leve) -->
#     no lugar da tag. Busca o JSON com fetch(). Só funciona em servidor / GitHub Pages.
#     Se abrir com dois cliques, mostra o aviso amarelo "Não consegui carregar o data.json automaticamente".
#   - index_standalone.html (STANDALONE): mesma base + a tag
#     <script id="embedded-data" type="application/json"> com o JSON minificado no lugar do comentário.
#     Já tem os dados dentro. Não precisa do JSON ao lado. Para dois cliques / WhatsApp / e-mail.
#   - As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual.
#     A única diferença é onde o dado mora.
# O que esta célula espera:
#   - Você sobe o index.html LEVE atual (o que tem o comentário placeholder).
#   - Se você subir por engano o index_standalone.html (que tem a tag embedded-data),
#     ela tira a tag e recria a leve a partir dele, para não duplicar dados.
print("Selecione o index.html LEVE atual do projeto Brasil Olímpico")
print("(Se você só tem o index_standalone.html, pode subir ele também — a célula tira a tag e recria a leve)")
uploaded_html = files.upload()
nome_html = list(uploaded_html.keys())[0]
print(f"HTML recebido: {nome_html}")

html = open(nome_html, encoding="utf-8").read()
novo_dados = json.dumps(dados, ensure_ascii=False, separators=(",", ":"))
placeholder = "<!-- Dados carregados de dados_brasil_olimpico.json (versão web, sem cópia embutida para ficar leve) -->"
novo_tag = '<script id="embedded-data" type="application/json">\n' + novo_dados + '\n</script>'

# Passo 1: Garantir que estamos partindo da base LEVE (sem embedded-data)
# Caso A: já é a leve (tem o placeholder) — usa como está
if placeholder in html:
    html_leve = html
    print("✅ Achada a base leve (placeholder) — usando como está.")
# Caso B: é a standalone (tem a tag embedded-data) — tira a tag e põe o placeholder no lugar
elif re.search(r'<script id="embedded-data"[^>]*>.*?</script>', html, flags=re.S):
    html_leve, n = re.subn(r'<script id="embedded-data"[^>]*>.*?</script>', placeholder, html, count=1, flags=re.S)
    print(f"✅ Achada a standalone (tem embedded-data) — removida a tag e recriada a leve (n={n}).")
# Caso C: é um HTML muito antigo (ainda usa var Ac = ...) — tira o bloco var Ac e põe o placeholder
# ATENÇÃO: o regex simples var Ac = {...}; corta no primeiro "};", então aqui fazemos a remoção por contagem de chaves.
else:
    print("⚠️ Não achei nem o placeholder nem a tag embedded-data. Tentando remover var Ac antigo por contagem de chaves...")
    start = html.find("var Ac=")
    if start == -1:
        print("❌ Não achei nem placeholder, nem embedded-data, nem var Ac. Confira se o arquivo é o index.html do Brasil Olímpico.")
        html_leve = None
    else:
        i = start + len("var Ac=")
        while i < len(html) and html[i] in " \t\n\r":
            i += 1
        brace_start = i
        depth = 0
        in_str = None
        esc = False
        j = brace_start
        end = None
        while j < len(html):
            ch = html[j]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == in_str:
                    in_str = None
            else:
                if ch == '"' or ch == "'":
                    in_str = ch
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        k = j + 1
                        while k < len(html) and html[k] in " \t\n\r":
                            k += 1
                        end = k + 1 if k < len(html) and html[k] == ";" else j + 1
                        break
            j += 1
        if end is None:
            print("❌ Não consegui achar o fim do var Ac. Confira o arquivo.")
            html_leve = None
        else:
            html_leve = html[:start] + placeholder + html[end:]
            print("✅ Removido var Ac antigo por contagem de chaves — leve recriada.")

if html_leve is not None:
    # --- Correção do bug do Fe (nov/2026) ---
    # Por que isso existe: o bundle calcula os KPIs / gráficos UMA VEZ, quando a página carrega,
    # a partir de uma variável chamada Fe. Na leve antiga, Fe começava vazio (Fe = {}) e só depois
    # o fetch() colocava os dados. Mas o cálculo já tinha sido feito com 0, e não era refeito.
    # Resultado: no GitHub, sem caixa amarela mas com 0s; local, com caixa amarela, escolhia o JSON,
    # a caixa sumia mas os dados não apareciam.
    # Correção: Fe tem que ser carregado ANTES do cálculo.
    #   Leve: tenta sessionStorage (upload manual) e depois fetch SÍNCRONO (bloqueante) do JSON.
    #   Standalone: tenta sessionStorage e depois o embedded-data.
    #   filePicker: salva no sessionStorage e dá location.reload() — assim a página recarrega
    #   já com o dado certo antes do cálculo, em vez de só setar Fe e chamar __render().
    # O Colab aplica essa correção sozinho, mesmo se o template que você subiu estiver com o bug antigo.
    FE_LEVE = 'var Fe=(function(){try{var __s=sessionStorage.getItem("brasil_olimpico_json");if(__s){var __j0=JSON.parse(__s);if(__j0&&__j0.medalhas&&__j0.medalhas.length)return __j0;}}catch(__e0){}try{var __xhr=new XMLHttpRequest();__xhr.open("GET","dados_brasil_olimpico.json",false);__xhr.send(null);if(__xhr.status===200||__xhr.status===0){var __j1=JSON.parse(__xhr.responseText);if(__j1&&__j1.medalhas&&__j1.medalhas.length)return __j1;}}catch(__e1){}return {};})()'
    FE_STAND = 'var Fe=(function(){try{var __s=sessionStorage.getItem("brasil_olimpico_json");if(__s){var __j0=JSON.parse(__s);if(__j0&&__j0.medalhas&&__j0.medalhas.length)return __j0;}}catch(__e0){}try{var el=document.getElementById("embedded-data");if(el&&el.textContent.trim()){return JSON.parse(el.textContent);}}catch(e){}return {};})()'
    START_LEVE = 'async function __start(){var __has=Fe&&Fe.medalhas&&Fe.medalhas.length;if(__has){try{sessionStorage.removeItem("brasil_olimpico_json");}catch(__e){}__render();return;}var __w=document.createElement("div");__w.id="aviso-versao-leve";__w.style.cssText="background:#fef3c7;color:#92400e;border:1px solid #f59e0b;padding:12px 16px;margin:0;font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;position:relative;z-index:9999;";__w.innerHTML="<b>Não consegui carregar o data.json automaticamente</b><br>Você abriu a versão leve (index.html). Ela não funciona com dois cliques, porque ela busca o JSON com fetch() e o navegador bloqueia isso em file://. Para abrir com dois cliques, use o index_standalone.html (1 arquivo só, já com os dados dentro). Para usar esta versão leve, abra com servidor / GitHub Pages, com o JSON na mesma pasta.<br><br><b>Ou carregue o JSON manualmente:</b><br><input type=\"file\" id=\"filePicker\" accept=\".json,application/json\"><br><span id=\"filePickerErro\" style=\"color:#991b1b;\"></span>";var __root=document.getElementById("root");if(__root&&__root.parentNode){__root.parentNode.insertBefore(__w,__root);}else{document.body.insertBefore(__w,document.body.firstChild);}var __picker=document.getElementById("filePicker");if(__picker){__picker.addEventListener("change",async function(__ev){var __f=__ev.target.files&&__ev.target.files[0];if(!__f)return;try{var __txt=await __f.text();var __j2=JSON.parse(__txt);if(typeof norm==="function"){try{__j2=norm(__j2);}catch(__eNorm){}}if(!__j2||!__j2.medalhas||!__j2.medalhas.length){throw new Error("o arquivo não tem medalhas (verifique se é o dados_brasil_olimpico.json)");}try{sessionStorage.setItem("brasil_olimpico_json",JSON.stringify(__j2));}catch(__eS){}location.reload();}catch(__err){var __eEl=document.getElementById("filePickerErro");if(__eEl){__eEl.textContent="Não consegui ler esse JSON: "+(__err&&__err.message?__err.message:String(__err));}}});}__render();}__start();'
    START_STAND = 'async function __start(){var __has=Fe&&Fe.medalhas&&Fe.medalhas.length;if(__has){try{sessionStorage.removeItem("brasil_olimpico_json");}catch(__e){}__render();return;}try{var __r2=await fetch("dados_brasil_olimpico.json?v="+Date.now(),{cache:"no-store"});if(__r2.ok){var __j3=await __r2.json();if(__j3&&__j3.medalhas&&__j3.medalhas.length){try{sessionStorage.setItem("brasil_olimpico_json",JSON.stringify(__j3));}catch(__eS){}location.reload();return;}}}catch(__e3){}var __w=document.createElement("div");__w.id="aviso-versao-leve";__w.style.cssText="background:#fef3c7;color:#92400e;border:1px solid #f59e0b;padding:12px 16px;margin:0;font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;position:relative;z-index:9999;";__w.innerHTML="<b>Não consegui carregar o data.json automaticamente</b><br>Não consegui carregar os dados embutidos desta versão standalone. Você pode carregar o dados_brasil_olimpico.json manualmente:<br><br><b>Ou carregue o JSON manualmente:</b><br><input type=\"file\" id=\"filePicker\" accept=\".json,application/json\"><br><span id=\"filePickerErro\" style=\"color:#991b1b;\"></span>";var __root=document.getElementById("root");if(__root&&__root.parentNode){__root.parentNode.insertBefore(__w,__root);}else{document.body.insertBefore(__w,document.body.firstChild);}var __picker=document.getElementById("filePicker");if(__picker){__picker.addEventListener("change",async function(__ev){var __f=__ev.target.files&&__ev.target.files[0];if(!__f)return;try{var __txt=await __f.text();var __j2=JSON.parse(__txt);if(typeof norm==="function"){try{__j2=norm(__j2);}catch(__eNorm){}}if(!__j2||!__j2.medalhas||!__j2.medalhas.length){throw new Error("o arquivo não tem medalhas (verifique se é o dados_brasil_olimpico.json)");}try{sessionStorage.setItem("brasil_olimpico_json",JSON.stringify(__j2));}catch(__eS){}location.reload();}catch(__err){var __eEl=document.getElementById("filePickerErro");if(__eEl){__eEl.textContent="Não consegui ler esse JSON: "+(__err&&__err.message?__err.message:String(__err));}}});}__render();}__start();'
    def _trocar_fe_e_start(html, fe_novo, start_novo):
        # Troca o Fe: qualquer var Fe=... até a vírgula que termina a definição, ou var Fe={} simples.
        # Padrões antigos: var Fe={}  |  var Fe=(function(){...embedded-data...})()  |  var Fe=(function(){...sync...})()
        # Usamos regex que pega var Fe= até o primeiro '})()' ou '{}' seguido de vírgula.
        # Para não quebrar, tentamos os padrões conhecidos na ordem.
        import re as _re
        orig = html
        # 1) var Fe={} simples
        if "var Fe={}" in html:
            html = html.replace("var Fe={}", fe_novo, 1)
        # 2) Fe antigo com embedded-data (standalone antiga)
        elif _re.search(r"var Fe=\(function\(\)\{try\{var el=document\.getElementById", html):
            html = _re.sub(r"var Fe=\(function\(\)\{try\{var el=document\.getElementById.*?return \{\};\}\)\(\)", fe_novo, html, count=1, flags=_re.S)
        # 3) Fe antigo com fetch async (qualquer outro Fe IIFE) — pega até o primeiro '})(),' que fecha o var
        elif "var Fe=(function()" in html:
            # Se já é o novo, não troca
            if fe_novo not in html:
                html = _re.sub(r"var Fe=\(function\(\)\{.*?return \{\};\}\)\(\)", fe_novo, html, count=1, flags=_re.S)
        # Troca o __start: pega async function __start(){...}__start(); até o __start(); final
        m = _re.search(r"async function __start\(\)\{.*?__render\(\);\}\__start\(\);", html, flags=_re.S)
        if m:
            html = html.replace(m.group(0), start_novo, 1)
        return html, (html != orig)
    # --- Melhorias de UX (Evolução + Estados + UX geral) ---
    # O Colab injeta sozinho essas 3 melhorias, mesmo se o template estiver sem elas.
    #   1) Evolução: destaque da barra + cartão branco (sem zoom, para o número não sair do gráfico).
    #   2) Estados: clique no mapa/tabela abre medalhistas (Nome - Modalidade), globo abre os 13 de fora.
    #   3) UX geral: destaque em todos os cards (os de cima SEM mexer na borda colorida, igual aos da Lista de medalhas),
    #      linhas das tabelas, nomes dos atletas, botões do cabeçalho, tooltips de Gênero e Idade
    #      (Idade: 'Medalhistas 222 (X% do total)' — sem a palavra Share).
    ENH_EVO = r''' <style id="evo-hover-style">
  #evolucao div[role="img"] > div { transition: opacity .15s ease, transform .15s ease; transform-origin: bottom; }
  #evolucao.evo-hovering div[role="img"] > div { opacity: .28; }
  #evolucao div[role="img"] > div.evo-active { opacity: 1 !important; transform: none; }
  #evolucao div[role="img"] > div.evo-active > div > div { box-shadow: 0 0 0 2px rgba(45,74,50,.25); }
  #evo-tooltip { position: fixed; z-index: 9999; pointer-events: none; background: #fff; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 28px rgba(0,0,0,.18); padding: 10px 12px; font-size: 13px; line-height: 1.45; min-width: 168px; max-width: 240px; display: none; }
  #evo-tooltip .evo-tt-title { font-weight: 800; font-size: 14px; color: #2D4A32; }
  #evo-tooltip .evo-tt-city { color: #6b7280; font-size: 12px; margin-bottom: 6px; }
  #evo-tooltip .evo-tt-row { display: flex; justify-content: space-between; gap: 12px; padding: 2px 0; }
  #evo-tooltip .evo-tt-row b { font-variant-numeric: tabular-nums; }
  #evo-tooltip .evo-tt-total { border-top: 1px solid #e5e7eb; margin-top: 6px; padding-top: 6px; font-weight: 800; }
  #evo-tooltip .evo-dot { display: inline-block; width: 9px; height: 9px; border-radius: 2px; margin-right: 6px; vertical-align: middle; }
</style>
<script id="evo-hover-script">
(function(){
  function initEvoHover(){
    var sec = document.getElementById("evolucao");
    if(!sec) return;
    var tip = document.getElementById("evo-tooltip");
    if(!tip){ tip = document.createElement("div"); tip.id = "evo-tooltip"; document.body.appendChild(tip); }
    function getCols(){ var chart = sec.querySelector('div[role="img"]'); if(!chart) return []; return Array.prototype.slice.call(chart.children); }
    function parseTitle(title){
      var m = title.match(/^(.+?)\s—\s(.+?):\s(\d+) ouro,\s(\d+) prata,\s(\d+) bronze\s\(total\s(\d+)\)/);
      if(m) return {ano:m[1], cidade:m[2], ouro:+m[3], prata:+m[4], bronze:+m[5], total:+m[6]};
      return null;
    }
    function showTip(e, col){
      var bar = col.querySelector('div[title]');
      var d = bar ? parseTitle(bar.getAttribute("title")||"") : null;
      if(!d){
        var cols = getCols(); var idx = cols.indexOf(col);
        var labels = sec.querySelectorAll('div.flex.mt-1 > div'); var lab = labels[idx];
        var ano = "", cidade = "";
        if(lab){ var parts = lab.querySelectorAll("div"); if(parts[0]) ano = parts[0].textContent.trim(); if(parts[1]) cidade = parts[1].textContent.trim(); }
        var totalEl = col.querySelector("div.text-center"); var total = totalEl ? parseInt(totalEl.textContent.trim(),10)||0 : 0;
        d = {ano:ano, cidade:cidade, ouro:0, prata:0, bronze:0, total:total};
      }
      tip.innerHTML = '<div class="evo-tt-title">'+d.ano+'</div>'
        + '<div class="evo-tt-city">'+d.cidade+'</div>'
        + '<div class="evo-tt-row"><span><span class="evo-dot" style="background:#FFD700"></span>Ouros</span><b>'+d.ouro+'</b></div>'
        + '<div class="evo-tt-row"><span><span class="evo-dot" style="background:#C0C0C0"></span>Pratas</span><b>'+d.prata+'</b></div>'
        + '<div class="evo-tt-row"><span><span class="evo-dot" style="background:#CD7F32"></span>Bronzes</span><b>'+d.bronze+'</b></div>'
        + '<div class="evo-tt-row evo-tt-total"><span>Total</span><b>'+d.total+' medalha'+(d.total===1?'':'s')+'</b></div>';
      tip.style.display = "block"; positionTip(e);
    }
    function positionTip(e){
      var pad = 14; var w = tip.offsetWidth, h = tip.offsetHeight;
      var x = e.clientX + pad, y = e.clientY + pad;
      if(x + w > window.innerWidth - 8) x = e.clientX - w - pad;
      if(y + h > window.innerHeight - 8) y = e.clientY - h - pad;
      tip.style.left = x + "px"; tip.style.top = y + "px";
    }
    function hideTip(){ tip.style.display = "none"; sec.classList.remove("evo-hovering"); getCols().forEach(function(c){ c.classList.remove("evo-active"); }); }
    sec.addEventListener("mouseover", function(e){
      var chart = sec.querySelector('div[role="img"]'); if(!chart || !chart.contains(e.target)) return;
      var col = e.target.closest('div[role="img"] > div'); if(!col) return;
      if(col.classList.contains("evo-active")){ positionTip(e); return; }
      sec.classList.add("evo-hovering"); getCols().forEach(function(c){ c.classList.toggle("evo-active", c===col); }); showTip(e, col);
    });
    sec.addEventListener("mousemove", function(e){ if(tip.style.display==="block") positionTip(e); });
    sec.addEventListener("mouseout", function(e){
      var to = e.relatedTarget;
      if(to && sec.contains(to)){ var col = to.closest && to.closest('div[role="img"] > div'); if(col) return; }
      hideTip();
    });
    sec.addEventListener("click", function(e){
      var col = e.target.closest && e.target.closest('div[role="img"] > div'); if(!col) return;
      sec.classList.add("evo-hovering"); getCols().forEach(function(c){ c.classList.toggle("evo-active", c===col); }); showTip(e, col);
    });
    document.addEventListener("click", function(e){ if(!sec.contains(e.target) && tip.style.display==="block") hideTip(); });
    // No celular: deslizar ou tocar fora esconde o tooltip (mouseout não dispara no touch)
    window.addEventListener("scroll", hideTip, {passive:true});
    document.addEventListener("touchmove", hideTip, {passive:true});
    document.addEventListener("touchstart", function(e){ if(!sec.contains(e.target)) hideTip(); }, {passive:true});
  }
  if(document.readyState==="loading") document.addEventListener("DOMContentLoaded", initEvoHover); else initEvoHover();
  setTimeout(initEvoHover, 500); setTimeout(initEvoHover, 1500);
})();
</script> '''
    ENH_ESTADO = r''' <style id="estado-detalhe-style">
  #estados .mapa-main svg path { cursor: pointer; transition: opacity .15s, stroke-width .15s; }
  #estados .mapa-main svg path:hover { opacity: .75; stroke-width: 1.4; }
  #estados .mapa-main svg g.estado-selecionado path { stroke: #2D4A32; stroke-width: 2; }
  #estados .globo-box { cursor: pointer; }
  #estados .globo-box:hover { opacity: .85; }
  #estados .quadro-tabela tbody tr { cursor: pointer; }
  #estados .quadro-tabela tbody tr:hover { background: #F9FAFB; }
  #estados .quadro-tabela tbody tr.estado-selecionado { background: #FEF9C3; }
  #estado-detalhe { margin-top: 16px; background: #fff; border: 1px solid var(--line, #e5e7eb); border-radius: 16px; padding: 16px; }
  #estado-detalhe h4 { font-weight: 800; font-size: 17px; color: #2D4A32; margin: 0 0 4px 0; }
  #estado-detalhe .estado-detalhe-sub { font-size: 13px; color: #6b7280; margin-bottom: 12px; }
  #estado-detalhe .estado-detalhe-lista { display: flex; flex-direction: column; gap: 0; max-height: 420px; overflow-y: auto; }
  #estado-detalhe .estado-detalhe-row { display: flex; align-items: center; gap: 10px; padding: 9px 4px; border-top: 1px solid #f3f4f6; font-size: 14px; }
  #estado-detalhe .estado-detalhe-pos { width: 28px; flex-shrink: 0; font-weight: 800; color: #2D4A32; }
  #estado-detalhe .estado-detalhe-nome { flex: 1; min-width: 0; font-weight: 600; }
  #estado-detalhe .estado-detalhe-cidade { font-size: 12px; color: #6b7280; font-weight: 400; }
  #estado-detalhe .estado-detalhe-med { display: flex; gap: 10px; font-size: 13px; font-variant-numeric: tabular-nums; white-space: nowrap; }
  #estado-detalhe .estado-detalhe-fechar { float: right; border: 0; background: transparent; cursor: pointer; font-size: 13px; font-weight: 700; color: #2D4A32; text-decoration: underline; }
</style>
<script id="estado-detalhe-script">
(function(){
  var SIGLA_ORDEM = ["RO","AC","AM","RR","PA","AP","TO","MA","PI","CE","RN","PB","PE","AL","SE","BA","MG","ES","RJ","SP","PR","SC","RS","MS","MT","GO","DF"];
  var NOME_PARA_SIGLA = {"Rondônia":"RO","Acre":"AC","Amazonas":"AM","Roraima":"RR","Pará":"PA","Amapá":"AP","Tocantins":"TO","Maranhão":"MA","Piauí":"PI","Ceará":"CE","Rio Grande do Norte":"RN","Paraíba":"PB","Pernambuco":"PE","Alagoas":"AL","Sergipe":"SE","Bahia":"BA","Minas Gerais":"MG","Espírito Santo":"ES","Rio de Janeiro":"RJ","São Paulo":"SP","Paraná":"PR","Santa Catarina":"SC","Rio Grande do Sul":"RS","Mato Grosso do Sul":"MS","Mato Grosso":"MT","Goiás":"GO","Distrito Federal":"DF"};
  var SIGLA_PARA_NOME = {}; Object.keys(NOME_PARA_SIGLA).forEach(function(n){ SIGLA_PARA_NOME[NOME_PARA_SIGLA[n]] = n; });
  function getRanking(){
    try{ var f = window.Fe || (typeof Fe!=="undefined" ? Fe : null); if(f && f.ranking) return f.ranking; }catch(e){}
    try{ var el = document.getElementById("embedded-data"); if(el && el.textContent.trim()){ var j = JSON.parse(el.textContent); if(j && j.ranking) return j.ranking; } }catch(e){}
    try{ var s = sessionStorage.getItem("brasil_olimpico_json"); if(s){ var j2 = JSON.parse(s); if(j2 && j2.ranking) return j2.ranking; } }catch(e){}
    return [];
  }
  function getModalidadeMap(){
    var mapa = {};
    try{
      var f = window.Fe || (typeof Fe!=="undefined" ? Fe : null);
      var medalhas = f && f.medalhas ? f.medalhas : null;
      if(!medalhas){ var el = document.getElementById("embedded-data"); if(el && el.textContent.trim()){ var j = JSON.parse(el.textContent); if(j && j.medalhas) medalhas = j.medalhas; } }
      if(!medalhas){ var s = sessionStorage.getItem("brasil_olimpico_json"); if(s){ var j2 = JSON.parse(s); if(j2 && j2.medalhas) medalhas = j2.medalhas; } }
      if(medalhas){ medalhas.forEach(function(m){ var mod = m.modalidade || ""; if(!mod) return; (m.atletas_detalhe||[]).forEach(function(a){ if(a && a.nome && !mapa[a.nome]) mapa[a.nome] = mod; }); (m.atletas||[]).forEach(function(n){ if(typeof n==="string" && n && !mapa[n]) mapa[n] = mod; }); }); }
    }catch(e){}
    return mapa;
  }
  function initEstadoDetalhe(){
    var sec = document.getElementById("estados"); if(!sec) return;
    if(document.getElementById("estado-detalhe")) return;
    var painel = document.createElement("div"); painel.id = "estado-detalhe";
    painel.innerHTML = '<button type="button" class="estado-detalhe-fechar" style="display:none">Fechar ✕</button>'
      + '<h4>Medalhistas por estado</h4>'
      + '<div class="estado-detalhe-sub">Clique num estado no mapa ou na tabela para ver os medalhistas nascidos lá. Clique no globo para ver os nascidos fora do Brasil.</div>'
      + '<div class="estado-detalhe-lista"></div>';
    var card = sec.querySelector("div.bg-\\[var\\(--card\\)\\]");
    if(card) card.appendChild(painel); else sec.appendChild(painel);
    var titulo = painel.querySelector("h4"); var sub = painel.querySelector(".estado-detalhe-sub");
    var lista = painel.querySelector(".estado-detalhe-lista"); var btnFechar = painel.querySelector(".estado-detalhe-fechar");
    function limparSelecao(){
      sec.querySelectorAll("g.estado-selecionado").forEach(function(g){ g.classList.remove("estado-selecionado"); });
      sec.querySelectorAll("tr.estado-selecionado").forEach(function(tr){ tr.classList.remove("estado-selecionado"); });
    }
    function mostrarEstado(sigla){
      var modMap = getModalidadeMap();
      var ranking = getRanking(); var nome = SIGLA_PARA_NOME[sigla] || sigla;
      var atletas = ranking.filter(function(a){ return NOME_PARA_SIGLA[a.estado] === sigla; })
        .slice().sort(function(a,b){ return (b.total||0)-(a.total||0) || (b.ouro||0)-(a.ouro||0) || (b.prata||0)-(a.prata||0) || (b.bronze||0)-(a.bronze||0) || String(a.nome||"").localeCompare(String(b.nome||"")); });
      limparSelecao();
      var gs = sec.querySelectorAll(".mapa-main svg g"); var mapaGs = []; gs.forEach(function(g){ mapaGs.push(g); });
      var idx = SIGLA_ORDEM.indexOf(sigla); if(idx>=0 && mapaGs[idx]) mapaGs[idx].classList.add("estado-selecionado");
      sec.querySelectorAll(".quadro-tabela tbody tr").forEach(function(tr){ var tds = tr.querySelectorAll("td"); if(tds[1] && tds[1].textContent.trim() === nome) tr.classList.add("estado-selecionado"); });
      titulo.textContent = "Medalhistas nascidos em " + nome + " (" + atletas.length + ")";
      sub.textContent = "Ordenados por total de medalhas. Clique em outro estado para trocar, ou em Fechar para limpar.";
      btnFechar.style.display = "";
      if(!atletas.length){ lista.innerHTML = '<div style="padding:12px 4px;color:#6b7280;font-size:14px">Nenhum medalhista nascido neste estado no conjunto de dados.</div>'; }
      else {
        lista.innerHTML = atletas.map(function(a, i){
          var cidade = a.cidade ? '<div class="estado-detalhe-cidade">'+a.cidade+'</div>' : '';
          return '<div class="estado-detalhe-row"><span class="estado-detalhe-pos">'+String(i+1).padStart(2,"0")+'</span>'
            + '<span class="estado-detalhe-nome">'+(a.nome||"") + (modMap[a.nome] ? " - " + modMap[a.nome] : "") + cidade + '</span>'
            + '<span class="estado-detalhe-med"><span>🥇 '+(a.ouro||0)+'</span><span>🥈 '+(a.prata||0)+'</span><span>🥉 '+(a.bronze||0)+'</span><b>'+(a.total||0)+' medalha'+((a.total||0)===1?'':'s')+'</b></span></div>';
        }).join("");
      }
      painel.scrollIntoView({behavior:"smooth", block:"nearest"});
    }
    function mostrarFora(){
      var modMap = getModalidadeMap();
      var ranking = getRanking();
      var fora = ranking.filter(function(a){ return a.pais && a.pais !== "Brasil"; })
        .slice().sort(function(a,b){ return (b.total||0)-(a.total||0) || String(a.nome||"").localeCompare(String(b.nome||"")); });
      limparSelecao();
      titulo.textContent = "Medalhistas nascidos fora do Brasil (" + fora.length + ")";
      sub.textContent = "Ordenados por total de medalhas."; btnFechar.style.display = "";
      if(!fora.length){ lista.innerHTML = '<div style="padding:12px 4px;color:#6b7280;font-size:14px">Nenhum medalhista nascido fora do Brasil.</div>'; }
      else {
        lista.innerHTML = fora.map(function(a, i){
          var loc = (a.cidade ? a.cidade + " — " : "") + (a.pais||"");
          return '<div class="estado-detalhe-row"><span class="estado-detalhe-pos">'+String(i+1).padStart(2,"0")+'</span>'
            + '<span class="estado-detalhe-nome">'+(a.nome||"") + (modMap[a.nome] ? " - " + modMap[a.nome] : "") + '<div class="estado-detalhe-cidade">'+loc+'</div></span>'
            + '<span class="estado-detalhe-med"><span>🥇 '+(a.ouro||0)+'</span><span>🥈 '+(a.prata||0)+'</span><span>🥉 '+(a.bronze||0)+'</span><b>'+(a.total||0)+' medalha'+((a.total||0)===1?'':'s')+'</b></span></div>';
        }).join("");
      }
      painel.scrollIntoView({behavior:"smooth", block:"nearest"});
    }
    btnFechar.addEventListener("click", function(){
      limparSelecao(); titulo.textContent = "Medalhistas por estado";
      sub.textContent = "Clique num estado no mapa ou na tabela para ver os medalhistas nascidos lá. Clique no globo para ver os nascidos fora do Brasil.";
      lista.innerHTML = ""; btnFechar.style.display = "none";
    });
    sec.addEventListener("click", function(e){
      var path = e.target.closest && e.target.closest(".mapa-main svg path");
      if(path){ var svg = path.closest("svg"); var paths = Array.prototype.slice.call(svg.querySelectorAll("path")); var idx = paths.indexOf(path); if(idx>=0 && idx < SIGLA_ORDEM.length){ mostrarEstado(SIGLA_ORDEM[idx]); return; } }
      var g = e.target.closest && e.target.closest(".mapa-main svg g");
      if(g){ var svg2 = g.closest("svg"); var gs = Array.prototype.slice.call(svg2.querySelectorAll("g")); var idx2 = gs.indexOf(g); if(idx2>=0 && idx2 < SIGLA_ORDEM.length){ mostrarEstado(SIGLA_ORDEM[idx2]); return; } }
      var globo = e.target.closest && e.target.closest(".globo-box"); if(globo){ mostrarFora(); return; }
      var tr = e.target.closest && e.target.closest(".quadro-tabela tbody tr");
      if(tr){ var tds = tr.querySelectorAll("td"); if(tds[1]){ var nome = tds[1].textContent.trim(); var sigla = NOME_PARA_SIGLA[nome]; if(sigla){ mostrarEstado(sigla); return; } } }
    });
  }
  if(document.readyState==="loading") document.addEventListener("DOMContentLoaded", initEstadoDetalhe); else initEstadoDetalhe();
  setTimeout(initEstadoDetalhe, 500); setTimeout(initEstadoDetalhe, 1500);
})();
</script> '''
    ENH_UX = r''' <style id="ux-geral-style">
  #root div.bg-\[var\(--card\)\] { transition: box-shadow .18s ease, border-color .18s ease, transform .18s ease; }
  #root div.bg-\[var\(--card\)\]:hover, #root div.bg-\[var\(--card\)\].ux-tapped { box-shadow: 0 8px 24px rgba(45,74,50,.12); border-color: #CBD5CB; transform: translateY(-1px); }
  #root .mini-card { transition: box-shadow .18s ease, border-color .18s ease, transform .18s ease; }
  #root .mini-card:hover, #root .mini-card.ux-tapped { box-shadow: 0 6px 18px rgba(45,74,50,.12); transform: translateY(-1px); }
  #inicio div.grid > div.rounded-2xl { transition: box-shadow .18s ease, transform .18s ease; cursor: default; }
  #inicio div.grid > div.rounded-2xl:hover, #inicio div.grid > div.rounded-2xl.ux-tapped { box-shadow: 0 8px 24px rgba(45,74,50,.14); transform: translateY(-2px); }
  #root table.quadro-tabela tbody tr { transition: background .15s ease; }
  #root table.quadro-tabela tbody tr:hover, #root table.quadro-tabela tbody tr.ux-tapped { background: #F3F8F3; }
  #medalhas article { transition: background .15s ease, box-shadow .15s ease; border-radius: 12px; padding-left: 8px; padding-right: 8px; margin-left: -8px; margin-right: -8px; }
  #medalhas article:hover, #medalhas article.ux-tapped { background: #F9FBF9; box-shadow: 0 2px 10px rgba(45,74,50,.08); }
  #medalhas article p span.font-medium { transition: color .15s ease; border-radius: 4px; padding: 0 2px; margin: 0 -2px; }
  #medalhas article p span.font-medium:hover, #medalhas article p span.font-medium.ux-tapped { color: #2D4A32; background: #FEF9C3; }
  #estado-detalhe .estado-detalhe-row { transition: background .15s ease; border-radius: 8px; }
  #estado-detalhe .estado-detalhe-row:hover, #estado-detalhe .estado-detalhe-row.ux-tapped { background: #F3F8F3; }
  #estado-detalhe .estado-detalhe-nome:hover { color: #2D4A32; }
  header nav a { transition: background .15s ease, transform .15s ease, box-shadow .15s ease !important; }
  header nav a:hover, header nav a.ux-tapped { background: rgba(255,255,255,.45) !important; transform: translateY(-1px); box-shadow: 0 2px 8px rgba(0,0,0,.10); }
  header div.lg\:hidden a { transition: background .15s ease, transform .15s ease, box-shadow .15s ease !important; }
  header div.lg\:hidden a:hover, header div.lg\:hidden a.ux-tapped { background: rgba(255,255,255,.45) !important; transform: translateY(-1px); }
  #root button.inline-flex:hover { box-shadow: 0 2px 8px rgba(45,74,50,.10); }
  .ux-tooltip { position: fixed; z-index: 9999; pointer-events: none; background: #fff; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 28px rgba(0,0,0,.18); padding: 10px 12px; font-size: 13px; line-height: 1.45; min-width: 168px; max-width: 240px; display: none; }
  .ux-tooltip .ux-tt-title { font-weight: 800; font-size: 14px; color: #2D4A32; }
  .ux-tooltip .ux-tt-city { color: #6b7280; font-size: 12px; margin-bottom: 6px; }
  .ux-tooltip .ux-tt-row { display: flex; justify-content: space-between; gap: 12px; padding: 2px 0; }
  .ux-tooltip .ux-tt-row b { font-variant-numeric: tabular-nums; }
  .ux-tooltip .ux-tt-total { border-top: 1px solid #e5e7eb; margin-top: 6px; padding-top: 6px; font-weight: 800; }
  .ux-tooltip .ux-dot { display: inline-block; width: 9px; height: 9px; border-radius: 2px; margin-right: 6px; vertical-align: middle; }
  #genero div[role="img"] > div { transition: opacity .15s ease; }
  #genero.ux-hovering div[role="img"] > div { opacity: .28; }
  #genero div[role="img"] > div.ux-active { opacity: 1 !important; }
  #genero div[role="img"] > div.ux-active > div > div { box-shadow: 0 0 0 2px rgba(45,74,50,.25); }
  #idade div.flex.items-end.gap-3 > div { transition: opacity .15s ease; }
  #idade.ux-hovering div.flex.items-end.gap-3 > div { opacity: .30; }
  #idade div.flex.items-end.gap-3 > div.ux-active { opacity: 1 !important; }
  #idade div.flex.items-end.gap-3 > div.ux-active > div { box-shadow: 0 0 0 2px rgba(45,74,50,.25); }
</style>
<script id="ux-geral-script">
(function(){
  function addTapped(el){ if(!el) return; el.classList.add("ux-tapped"); setTimeout(function(){ el.classList.remove("ux-tapped"); }, 400); }
  document.addEventListener("click", function(e){
    var card = e.target.closest && e.target.closest('#root div.bg-\\[var\\(--card\\)\\], #root .mini-card, #inicio div.grid > div.rounded-2xl');
    if(card) addTapped(card);
    var tr = e.target.closest && e.target.closest('#root table.quadro-tabela tbody tr'); if(tr) addTapped(tr);
    var art = e.target.closest && e.target.closest('#medalhas article'); if(art) addTapped(art);
    var row = e.target.closest && e.target.closest('#estado-detalhe .estado-detalhe-row'); if(row) addTapped(row);
    var nome = e.target.closest && e.target.closest('#medalhas article p span.font-medium'); if(nome) addTapped(nome);
    var navA = e.target.closest && e.target.closest('header nav a, header div.lg\\:hidden a'); if(navA) addTapped(navA);
  }, true);
  var tip = null;
  function getTip(){ if(!tip){ tip = document.createElement("div"); tip.className = "ux-tooltip"; tip.id = "ux-tooltip"; document.body.appendChild(tip); } return tip; }
  function positionTip(e){
    var t = getTip(); var pad = 14; var w = t.offsetWidth, h = t.offsetHeight;
    var x = e.clientX + pad, y = e.clientY + pad;
    if(x + w > window.innerWidth - 8) x = e.clientX - w - pad;
    if(y + h > window.innerHeight - 8) y = e.clientY - h - pad;
    t.style.left = x + "px"; t.style.top = y + "px";
  }
  function hideTip(){ var t = getTip(); t.style.display = "none"; }
  function hideUxAll(){
    hideTip();
    ["genero","idade"].forEach(function(id){
      var s = document.getElementById(id); if(!s) return;
      s.classList.remove("ux-hovering");
      s.querySelectorAll(".ux-active").forEach(function(c){ c.classList.remove("ux-active"); });
    });
  }
  function initGenero(){
    var sec = document.getElementById("genero"); if(!sec) return;
    function getCols(){ var chart = sec.querySelector('div[role="img"]'); if(!chart) return []; return Array.prototype.slice.call(chart.children); }
    function parseTitle(title){
      var m = title.match(/^(.+?)\s—\s(.+?):\sMasculino\s(\d+),\sFeminino\s(\d+),\sMisto\s(\d+)\s\(total\s(\d+)\)/);
      if(m) return {ano:m[1], cidade:m[2], masc:+m[3], fem:+m[4], misto:+m[5], total:+m[6]}; return null;
    }
    function renderTip(d){
      var t = getTip();
      t.innerHTML = '<div class="ux-tt-title">'+d.ano+'</div>'
        + '<div class="ux-tt-city">'+d.cidade+'</div>'
        + '<div class="ux-tt-row"><span><span class="ux-dot" style="background:#93C5FD"></span>Masculino</span><b>'+d.masc+'</b></div>'
        + '<div class="ux-tt-row"><span><span class="ux-dot" style="background:#F9A8D4"></span>Feminino</span><b>'+d.fem+'</b></div>'
        + '<div class="ux-tt-row"><span><span class="ux-dot" style="background:#C4B5FD"></span>Misto</span><b>'+d.misto+'</b></div>'
        + '<div class="ux-tt-row ux-tt-total"><span>Total</span><b>'+d.total+' medalha'+(d.total===1?'':'s')+'</b></div>';
      t.style.display = "block";
    }
    sec.addEventListener("mouseover", function(e){
      var chart = sec.querySelector('div[role="img"]'); if(!chart || !chart.contains(e.target)) return;
      var col = e.target.closest('div[role="img"] > div'); if(!col) return;
      if(col.classList.contains("ux-active")){ positionTip(e); return; }
      sec.classList.add("ux-hovering"); getCols().forEach(function(c){ c.classList.toggle("ux-active", c===col); });
      var bar = col.querySelector('div[title]'); var d = bar ? parseTitle(bar.getAttribute("title")||"") : null; if(!d) return; renderTip(d); positionTip(e);
    });
    sec.addEventListener("mousemove", function(e){ var t = getTip(); if(t.style.display==="block") positionTip(e); });
    sec.addEventListener("mouseout", function(e){
      var to = e.relatedTarget; if(to && sec.contains(to)){ var col = to.closest && to.closest('div[role="img"] > div'); if(col) return; }
      sec.classList.remove("ux-hovering"); getCols().forEach(function(c){ c.classList.remove("ux-active"); }); hideTip();
    });
    sec.addEventListener("click", function(e){
      var col = e.target.closest && e.target.closest('div[role="img"] > div'); if(!col) return;
      sec.classList.add("ux-hovering"); getCols().forEach(function(c){ c.classList.toggle("ux-active", c===col); });
      var bar = col.querySelector('div[title]'); var d = bar ? parseTitle(bar.getAttribute("title")||"") : null; if(!d) return; renderTip(d); positionTip(e);
    });
  }
  function initIdade(){
    var sec = document.getElementById("idade"); if(!sec) return;
    function getBars(){ var chart = sec.querySelector('div.flex.items-end.gap-3'); if(!chart) return []; return Array.prototype.slice.call(chart.children); }
    function showFor(bar, e){
      var divs = bar.querySelectorAll(":scope > div"); if(divs.length < 3) return;
      var count = parseInt((divs[0].textContent||"").trim(), 10) || 0;
      var label = (divs[2].textContent||"").trim();
      var bars = getBars(); var total = 0;
      bars.forEach(function(b){ var c = b.querySelector(":scope > div"); if(c) total += parseInt((c.textContent||"").trim(), 10) || 0; });
      var pct = total ? Math.round(count/total*100) : 0;
      var t = getTip();
      t.innerHTML = '<div class="ux-tt-title">Faixa '+label+' anos</div>'
        + '<div class="ux-tt-city">Idade dos medalhistas</div>'
        + '<div class="ux-tt-row"><span>Medalhistas</span><b>'+count+' ('+pct+'% do total)</b></div>'
        + '<div class="ux-tt-row ux-tt-total"><span>Total</span><b>'+total+' medalhistas</b></div>';
      t.style.display = "block"; positionTip(e);
    }
    sec.addEventListener("mouseover", function(e){
      var chart = sec.querySelector('div.flex.items-end.gap-3'); if(!chart || !chart.contains(e.target)) return;
      var bar = e.target.closest('div.flex.items-end.gap-3 > div'); if(!bar) return;
      if(bar.classList.contains("ux-active")){ positionTip(e); return; }
      sec.classList.add("ux-hovering"); getBars().forEach(function(b){ b.classList.toggle("ux-active", b===bar); }); showFor(bar, e);
    });
    sec.addEventListener("mousemove", function(e){ var t = getTip(); if(t.style.display==="block") positionTip(e); });
    sec.addEventListener("mouseout", function(e){
      var to = e.relatedTarget; if(to && sec.contains(to)){ var bar = to.closest && to.closest('div.flex.items-end.gap-3 > div'); if(bar) return; }
      sec.classList.remove("ux-hovering"); getBars().forEach(function(b){ b.classList.remove("ux-active"); }); hideTip();
    });
    sec.addEventListener("click", function(e){
      var bar = e.target.closest && e.target.closest('div.flex.items-end.gap-3 > div'); if(!bar) return;
      sec.classList.add("ux-hovering"); getBars().forEach(function(b){ b.classList.toggle("ux-active", b===bar); }); showFor(bar, e);
    });
  }
  function init(){ initGenero(); initIdade();
    // No celular: deslizar ou tocar fora esconde o tooltip (mouseout não dispara no touch)
    window.addEventListener("scroll", hideUxAll, {passive:true});
    document.addEventListener("touchmove", hideUxAll, {passive:true});
    document.addEventListener("touchstart", function(e){
      var g = document.getElementById("genero"); var idd = document.getElementById("idade");
      var inside = (g && g.contains(e.target)) || (idd && idd.contains(e.target));
      if(!inside) hideUxAll();
    }, {passive:true});
  }
  if(document.readyState==="loading") document.addEventListener("DOMContentLoaded", init); else init();
  setTimeout(init, 500); setTimeout(init, 1500);
})();
</script> '''
    def _injetar_melhorias(html):
        for _key, _enh, _nome in [('evo-hover-script', ENH_EVO, 'Evolução'), ('estado-detalhe-script', ENH_ESTADO, 'Estados'), ('ux-geral-script', ENH_UX, 'UX geral')]:
            if _key not in html:
                _idx = html.rfind('</body>')
                if _idx != -1:
                    html = html[:_idx] + _enh + '\n' + html[_idx:]
                    print(f'✅ Melhoria {_nome} injetada (template estava sem).')
                else:
                    print(f'⚠️ Não achei </body> para injetar a melhoria {_nome}.')
        return html
    html_leve = _injetar_melhorias(html_leve)
    html_leve, mudou = _trocar_fe_e_start(html_leve, FE_LEVE, START_LEVE)
    print(f"✅ Correção do Fe aplicada na leve: {'sim (template estava com bug antigo)' if mudou else 'já estava corrigida'}")
    # Garantir que a leve NÃO tenha tag embedded-data (por segurança, se sobrou alguma)
    if re.search(r'<script id="embedded-data"', html_leve):
        print("⚠️ A leve ainda tinha uma tag embedded-data — removendo por segurança.")
        html_leve, _ = re.subn(r'<script id="embedded-data"[^>]*>.*?</script>', placeholder, html_leve, count=1, flags=re.S)
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_leve)
    print(f"✅ Gerado index.html (LEVE, sem dados embutidos) com {len(medalhas)} medalhas no JSON ao lado.")

    # Passo 3: A standalone é a base + a tag embedded-data com o JSON minificado no lugar do placeholder
    # ATENÇÃO: a standalone precisa do Fe de standalone (sessionStorage + embedded-data), não do Fe de leve (sync XHR).
    # Se usarmos o Fe de leve na standalone, ela tentaria o XHR e ignoraria o embedded-data no file://.
    if placeholder not in html_leve:
        print("❌ Não achei o placeholder na leve para gerar a standalone. Confira o arquivo.")
        html_standalone = None
    else:
        html_standalone = html_leve.replace(placeholder, novo_tag, count=1)
        html_standalone, mudou_s = _trocar_fe_e_start(html_standalone, FE_STAND, START_STAND)
        print(f"✅ Correção do Fe aplicada na standalone: {'sim' if mudou_s else 'já estava corrigida'}")
        with open("index_standalone.html", "w", encoding="utf-8") as f:
            f.write(html_standalone)
        print(f"✅ Gerado index_standalone.html (STANDALONE, 1 arquivo só, com os dados dentro) com {len(medalhas)} medalhas embutidas.")
        print(f"   As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual.")


In [ ]:
# --- CÉLULA 6: Baixar os 3 arquivos atualizados ---
files.download("dados_brasil_olimpico.json")
files.download("index.html")
files.download("index_standalone.html")
print("Baixe os 3 arquivos e suba os 3 no GitHub no lugar dos antigos. Pronto!")
print("GitHub Pages: use index.html (leve) + dados_brasil_olimpico.json na mesma pasta.")
print("Dois cliques / WhatsApp / e-mail: use index_standalone.html (1 arquivo só, já com os dados dentro).")
